## 09.03 深度循环神经网络


### 环境配置


In [1]:
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    from torch.nn import functional as F
    import torch_npu
    import logging

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor")
logging.getLogger("torch_npu").setLevel(logging.WARNING)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

### 练习 9.3.1

**题目：** 基于 8.5 节单层实现，从零开始实现两层循环神经网络。

**解答：** 定义两层 RNN 的参数和状态，每层独立计算隐状态递推。与官方手册一致，此处做了简化：第二层的输入直接取第一层的隐状态 $H_1$（未引入独立的 $W_{xh2}$），即 $H_2 = \text{relu}(H_1 W_{hh2} + b_{h2})$。

以下使用 `torch` 编程：



In [2]:
import math
import torch
from torch import nn
from torch.nn import functional as F
from src.utils import load_data_time_machine, train_ch8, try_gpu, RNNModelScratch

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape): return torch.randn(size=shape, device=device) * 0.01
    W_xh1 = normal((num_inputs, num_hiddens))
    W_hh1 = normal((num_hiddens, num_hiddens))
    b_h1 = torch.zeros(num_hiddens, device=device)
    W_xh2 = normal((num_hiddens, num_hiddens))
    W_hh2 = normal((num_hiddens, num_hiddens))
    b_h2 = torch.zeros(num_hiddens, device=device)
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xh1, W_hh1, b_h1, W_xh2, W_hh2, b_h2, W_hq, b_q]
    for param in params: param.requires_grad_(True)
    return params

def init_rnn_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),
            torch.zeros((batch_size, num_hiddens), device=device))

def rnn(inputs, state, params):
    W_xh1, W_hh1, b_h1, W_xh2, W_hh2, b_h2, W_hq, b_q = params
    H1, H2 = state
    outputs = []
    for X in inputs:
        H1 = torch.relu(torch.mm(X, W_xh1) + torch.mm(H1, W_hh1) + b_h1)
        H2 = torch.relu(torch.mm(H1, W_xh2) + torch.mm(H2, W_hh2) + b_h2)
        Y = torch.mm(H2, W_hq) + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H1, H2)

num_hiddens = 512
net = RNNModelScratch(len(vocab), num_hiddens, try_gpu(),
                      get_params, init_rnn_state, rnn)
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, try_gpu(), use_plot=False)

使用 `PyPTO` 编程：



In [3]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import (PyPTOMatmul, PyPTOBiasAdd, PyPTOAdd,
                           PyPTOReLUOp, loss_fn)

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

def get_params_pypto(vocab_size, num_hiddens, device):
    def normal(shape): return torch.randn(size=shape, device=device) * 0.01
    num_inputs = num_outputs = vocab_size
    w = [normal((num_inputs, num_hiddens)), normal((num_hiddens, num_hiddens)),
         torch.zeros(num_hiddens, device=device),
         normal((num_hiddens, num_hiddens)), normal((num_hiddens, num_hiddens)),
         torch.zeros(num_hiddens, device=device),
         normal((num_hiddens, num_outputs)), torch.zeros(num_outputs, device=device)]
    for p in w: p.requires_grad_(True)
    return w

def rnn_pypto(inputs, state, params):
    W_xh1, W_hh1, b_h1, W_xh2, W_hh2, b_h2, W_hq, b_q = params
    H1, H2 = state
    outputs = []
    for X in inputs:
        XW1 = PyPTOMatmul.apply(X, W_xh1)
        HW1 = PyPTOMatmul.apply(H1, W_hh1)
        H1 = PyPTOReLUOp.apply(PyPTOBiasAdd.apply(PyPTOAdd.apply(XW1, HW1), b_h1))
        XW2 = PyPTOMatmul.apply(H1, W_xh2)
        HW2 = PyPTOMatmul.apply(H2, W_hh2)
        H2 = PyPTOReLUOp.apply(PyPTOBiasAdd.apply(PyPTOAdd.apply(XW2, HW2), b_h2))
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H2, W_hq), b_q)
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H1, H2)

class PyPTORNNModelScratch:
    def __init__(self, vocab_size, num_hiddens, device, get_params_fn, init_state_fn, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.device = device
        self.params = get_params_fn(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state_fn, forward_fn
    def __call__(self, X, state):
        X_oh = F.one_hot(X.T, self.vocab_size).float()
        return self.forward_fn(X_oh, state, self.params)
    def begin_state(self, batch_size, device=None):
        d = device or self.device
        return self.init_state(batch_size, self.num_hiddens, d)

X, y = next(iter(train_iter))
net_w = PyPTORNNModelScratch(len(vocab), 512, device, get_params_pypto,
    lambda b, h, d: (torch.zeros((b, h), device=d), torch.zeros((b, h), device=d)), rnn_pypto)
s_w = net_w.begin_state(batch_size, device)
l = loss_fn(net_w(X.to(device), s_w)[0], y.T.reshape(-1).to(device), len(vocab))
l.backward()

net = PyPTORNNModelScratch(len(vocab), 512, device, get_params_pypto,
    lambda b, h, d: (torch.zeros((b, h), device=d), torch.zeros((b, h), device=d)), rnn_pypto)
train_ch8(net, train_iter, vocab, 1, 500, device, use_plot=False, verbose=False,
          loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

困惑度 1.0, 9190.9 词元/秒 npu:0


time travelleryou can show black is white by argument said filby
travelleryou can show black is white by argument said filby


### 练习 9.3.2

**题目：** 比较使用 GRU 替换 LSTM 后模型的精确度和训练速度。

**解答：** 将 LSTM 替换为 GRU 后模型训练速度有所提高（GRU 只有两个门，参数更少），同时收敛更快。在简单文本上两者困惑度均可达到 1.0。

以下使用 `torch` 编程：



In [4]:
from src.utils import load_data_time_machine, train_ch8, try_gpu, RNNModel

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

vocab_size, num_hiddens, num_layers = len(vocab), 256, 2
device_d2l = try_gpu()

# LSTM
lstm_layer = nn.LSTM(vocab_size, num_hiddens, num_layers)
model_lstm = RNNModel(lstm_layer, len(vocab)).to(device_d2l)
num_epochs, lr = 500, 2
train_ch8(model_lstm, train_iter, vocab, lr, num_epochs, device_d2l,
          use_plot=False)

# GRU
gru_layer = nn.GRU(vocab_size, num_hiddens, num_layers)
model_gru = RNNModel(gru_layer, len(vocab)).to(device_d2l)
train_ch8(model_gru, train_iter, vocab, lr, num_epochs, device_d2l,
          use_plot=False)

使用 `PyPTO` 编程：

In [5]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import PyPTOLinear, PyPTOLSTM, PyPTOGRU, loss_fn

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

class PyPTORNNModelMulti(nn.Module):
    def __init__(self, rnn_layer, vocab_size):
        super().__init__()
        self.rnn = rnn_layer
        self.linear = PyPTOLinear(rnn_layer.hidden_size, vocab_size)
        self.vocab_size = vocab_size
    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size).float()
        Y, state = self.rnn(X, state)
        return self.linear(Y.reshape((-1, Y.shape[-1]))), state
    def begin_state(self, batch_size, device):
        if not isinstance(self.rnn, PyPTOLSTM):
            return torch.zeros((self.rnn.num_layers, batch_size, self.rnn.hidden_size), device=device)
        return (torch.zeros((self.rnn.num_layers, batch_size, self.rnn.hidden_size), device=device),
                torch.zeros((self.rnn.num_layers, batch_size, self.rnn.hidden_size), device=device))

X_w, y_w = next(iter(train_iter))
loss_fn_w = lambda y_h, y: loss_fn(y_h, y, len(vocab))

net_lstm = PyPTORNNModelMulti(PyPTOLSTM(len(vocab), 256, num_layers=2), len(vocab)).to(device)
s = net_lstm.begin_state(batch_size, device)
l = loss_fn_w(net_lstm(X_w.to(device), s)[0], y_w.T.reshape(-1).to(device))
l.backward(); net_lstm.zero_grad()
train_ch8(net_lstm, train_iter, vocab, 2, 500, device, use_plot=False, verbose=False, loss_fn=loss_fn_w)

net_gru = PyPTORNNModelMulti(PyPTOGRU(len(vocab), 256, num_layers=2), len(vocab)).to(device)
s2 = net_gru.begin_state(batch_size, device)
l2 = loss_fn_w(net_gru(X_w.to(device), s2)[0], y_w.T.reshape(-1).to(device))
l2.backward(); net_gru.zero_grad()
train_ch8(net_gru, train_iter, vocab, 2, 500, device, use_plot=False, verbose=False, loss_fn=loss_fn_w)

困惑度 1.0, 4742.8 词元/秒 npu:0


time traveller for so it will be convenient to speak of himwas e


traveller with a slight accession ofcheerfulness really thi


困惑度 1.0, 2466.3 词元/秒 npu:0


time travelleryou can show black is white by argument said filby


traveller with a slight accession ofcheerfulness really thi


### 练习 9.3.3

**题目：** 如果增加训练数据，能够将困惑度降到多低？

**解答：** 将《The Time Machine》与《The War of the Worlds》混合作为训练数据，困惑度仍可降到 1.0。（结论引自官方手册的混合数据集实验，此处不再重复训练。）


### 练习 9.3.4

**题目：** 为文本建模时，能否将不同作者的源数据合并？有何优劣？

**解答：** 
- 优点：增加训练数据量、提高模型泛化能力（学习不同风格、语言特征）
- 缺点：可能引入噪声影响训练效果、若作者数据不平衡可能引入偏见。

综上，合并前需考虑数据质量和平衡性，并做适当预处理。



---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)

